<a href="https://colab.research.google.com/github/nmwiley808/csci198-Music-Intelligence-with-Deep-Learning-Senior-Project/blob/main/notebooks/02%20-%20Audio%20Preprocessing%20%26%20Feature%20Extraction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 02 – Audio Preprocessing & Feature Extraction

This notebook converts raw audio from:

- GTZAN
- FMA Small
- MagnaTagATune

into standardized log-mel spectrogram features.

All audio is:
- Resampled to 22,050 Hz
- Standardized to 30 seconds
- Converted to log-mel spectrograms

Processed arrays are saved to `data/processed/` for model training.

In [1]:
# Mount Drive & Setup Enviroment
from google.colab import drive
drive.mount('/content/drive')

import os
import librosa
import numpy as np
import time
import tqdm

PROJECT_PATH = "/content/drive/MyDrive/csci198/csci198-Music-Intelligence-with-Deep-Learning-Senior-Project"
os.chdir(PROJECT_PATH)

print("Working directory: ", os.getcwd())

Mounted at /content/drive
Working directory:  /content/drive/MyDrive/csci198/csci198-Music-Intelligence-with-Deep-Learning-Senior-Project


In [2]:
# Define Standardization Parameters
SAMPLE_RATE = 22050
DURATION = 30
SAMPLES_PER_TRACK = SAMPLE_RATE * DURATION
N_MELS = 128

PROCESSED_PATH = "data/processed"
os.makedirs(PROCESSED_PATH, exist_ok=True)

In [3]:
# Load + Pad Function
def load_audio(file_path):
    try:
        y, sr = librosa.load(file_path, sr=SAMPLE_RATE)

        if len(y) < SAMPLES_PER_TRACK:
            y = np.pad(y, (0, SAMPLES_PER_TRACK - len(y)))
        else:
            y = y[:SAMPLES_PER_TRACK]

        return y

    except:
        return None

In [4]:
# Log-Mel Extraction
def extract_log_mel(y):
    mel = librosa.feature.melspectrogram(
        y=y,
        sr=SAMPLE_RATE,
        n_mels=N_MELS
    )
    log_mel = librosa.power_to_db(mel)
    return log_mel

In [5]:
# Process GTZAN
gtzan_path = "data/raw/gtzan/Data/genres_original"

X_gtzan = []
y_gtzan = []

print("Processing GTZAN...")
start = time.time()

for label in os.listdir(gtzan_path):
    genre_path = os.path.join(gtzan_path, label)
    if not os.path.isdir(genre_path):
        continue

    for file in tqdm.tqdm(os.listdir(genre_path)):
        file_path = os.path.join(genre_path, file)
        y_audio = load_audio(file_path)
        if y_audio is None:
            continue

        features = extract_log_mel(y_audio)
        X_gtzan.append(features)
        y_gtzan.append(label)

print("GTZAN Time:", round(time.time() - start, 2), "seconds")

Processing GTZAN...


 49%|████▉     | 49/100 [00:09<00:05,  8.87it/s]/tmp/ipykernel_1425/3775047078.py:4: UserWarning: PySoundFile failed. Trying audioread instead.
  y, sr = librosa.load(file_path, sr=SAMPLE_RATE)
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)
100%|██████████| 100/100 [00:24<00:00,  4.06it/s]

GTZAN Time: 226.67 seconds


In [6]:
X_gtzan = np.array(X_gtzan)
y_gtzan = np.array(y_gtzan)

np.save("data/processed/X_gtzan.npy", X_gtzan)
np.save("data/processed/y_gtzan.npy", y_gtzan)

print("GTZAN saved:", X_gtzan.shape)

GTZAN saved: (999, 128, 1292)


In [7]:
# Process FMA
fma_path = "data/raw/fma/fma_small"

X_fma = []
y_fma = []

print("Processing FMA...")
start = time.time()

for root, dirs, files in os.walk(fma_path):
    for file in tqdm.tqdm(files):
        if not file.endswith(".mp3"):
            continue

        file_path = os.path.join(root, file)
        y_audio = load_audio(file_path)
        if y_audio is None:
            continue

        features = extract_log_mel(y_audio)

        # Genre folder name
        label = os.path.basename(root)

        X_fma.append(features)
        y_fma.append(label)

print("FMA Time:", round(time.time() - start, 2), "seconds")

Processing FMA...


 29%|██▉       | 13/45 [00:11<00:28,  1.11it/s]/tmp/ipykernel_1425/3775047078.py:4: UserWarning: PySoundFile failed. Trying audioread instead.
  y, sr = librosa.load(file_path, sr=SAMPLE_RATE)
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)
 88%|████████▊ | 126/143 [00:50<00:12,  1.32it/s]/tmp/ipykernel_1425/3775047078.py:4: UserWarning: PySoundFile failed. Trying audioread instead.
  y, sr = librosa.load(file_path, sr=SAMPLE_RATE)
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)
 55%|█████▍    | 45/82 [00:41<00:33,  1.11it/s]/tmp/ipykernel_1425/3775047

FMA Time: 6618.18 seconds


In [3]:
# Save FMA
X_fma = np.array(X_fma)
y_fma = np.array(y_fma)

np.save("data/processed/X_fma.npy", X_fma)
np.save("data/processed/y_fma.npy", y_fma)

print("FMA saved:", X_fma.shape)

NameError: name 'X_fma' is not defined

In [ ]:
# Process MagnaTagATune
magna_path = "data/raw/magnatagatune"

X_magna = []
y_magna = []

print("Processing Magna...")
start = time.time()

for file in tqdm.tqdm(os.listdir(magna_path)):
    if not file.endswith(".wav"):
        continue

    file_path = os.path.join(magna_path, file)
    y_audio = load_audio(file_path)
    if y_audio is None:
        continue

    features = extract_log_mel(y_audio)

    X_magna.append(features)
    y_magna.append(file)

print("Magna Time:", round(time.time() - start, 2), "seconds")

In [ ]:
# Save Magna
X_magna = np.array(X_magna)
y_magna = np.array(y_magna)

np.save("data/processed/X_magna.npy", X_magna)
np.save("data/processed/y_magna.npy", y_magna)

print("Magna saved:", X_magna.shape)